# 07 - Pipeline Checks & Multi-Controller Evaluation on Held-Out Splits

This notebook verifies the updated experimental pipeline aligned with the project proposal and Week 11 requirements:

1. **Held-Out Dataset Splits**: Verifies `train` (June 1–20), `validation` (June 21–25), and `test` (June 26–30) intervals defined in `src.dataset_splits`.
2. **Decoupled Fairness Objectives**:
   - **Worst-Case Fair MARL** (Proposal RQ1): Penalizes the maximum thermal discomfort experienced across all buildings ($\max_i L_i$).
   - **Disparity-Aware MARL** (Proposal RQ2): Penalizes the spread between the most and least comfortable buildings ($\max_i L_i - \min_i L_i$).
3. **Multi-Controller Alignment on the Held-Out Test Split**:
   - Reference Baseline 1: Rule-Based Control (`BasicRBC`)
   - Reference Baseline 2: Independent RL (`Independent SAC`)
   - Coordinated MARL: `Standard Carbon-Aware MARL`
   - Fair Coordinated MARL: `Worst-Case Fair MARL`
   - Disparity Coordinated MARL: `Disparity-Aware MARL`
4. **Pipeline Fixes Compatibility**: Verifies support for simulated temperature observations and training-window observation encoders on held-out intervals.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_splits import TRAIN_SPLIT, VALIDATION_SPLIT, TEST_SPLIT, SPLITS
from src.rewards import (
    CarbonAwareReward,
    FairCarbonAwareReward,
    WorstCaseFairReward,
    DisparityAwareReward,
)
from src.run_marlisa import make_marlisa_env, evaluate_marlisa, run_marlisa
from src.run_fair_carbon_aware_marlisa import run_fair_carbon_aware_marlisa
from src.run_carbon_aware_marlisa import run_carbon_aware_marlisa
from src.run_independent_sac import make_sac_env

SCHEMA = PROJECT_ROOT / "notebooks" / "data" / "schema.json"
print("Project root:", PROJECT_ROOT)
print("Schema path:", SCHEMA)

Project root: /home/default/box/projects/cs5446_project
Schema path: /home/default/box/projects/cs5446_project/notebooks/data/schema.json


## 1. Held-Out Dataset Splits

CityLearn 2023 dataset covers 720 hours (June 1–30). The project defines three disjoint time splits:

In [2]:
splits_summary = pd.DataFrame([
    TRAIN_SPLIT.as_dict(),
    VALIDATION_SPLIT.as_dict(),
    TEST_SPLIT.as_dict(),
])
splits_summary

,name,start_time_step,end_time_step,start_date,end_date,n_time_steps,n_transitions
0,train,0,479,June 1,June 20,480,479
1,validation,480,599,June 21,June 25,120,119
2,test,600,719,June 26,June 30,120,119


## 2. Decoupled Fairness Reward Formulations

The project proposal asks two complementary research questions:
- **RQ1**: Can coordinated MARL limit discomfort in the *worst-affected* building?
- **RQ2**: How does a *disparity-minimizing* objective affect the trade-off compared to worst-case fairness?

Here we numerically verify that the decoupled reward functions correctly distinguish between worst-case loss and inter-building disparity.

In [3]:
# Building 0: High discomfort (comfort loss = 2.0, emission = 1.0)
# Building 1: Zero discomfort (comfort loss = 0.0, emission = 0.5)
obs_unequal = [
    {
        "net_electricity_consumption": 2.0,
        "carbon_intensity": 0.5,
        "indoor_dry_bulb_temperature": 26.0,
        "indoor_dry_bulb_temperature_cooling_set_point": 24.0,
        "occupant_count": 2.0,
    },
    {
        "net_electricity_consumption": 1.0,
        "carbon_intensity": 0.5,
        "indoor_dry_bulb_temperature": 24.0,
        "indoor_dry_bulb_temperature_cooling_set_point": 24.0,
        "occupant_count": 1.0,
    },
]

reward_wc = WorstCaseFairReward({"central_agent": False})
reward_disp = DisparityAwareReward({"central_agent": False})
reward_comb = FairCarbonAwareReward({"central_agent": False}, fairness_mode="combined")

print("=== Unequal Comfort Scenario ===")
print("Worst-Case Fair Rewards (penalizes max loss 2.0):", reward_wc.calculate(obs_unequal))
print("Disparity-Aware Rewards (penalizes gap 2.0):     ", reward_disp.calculate(obs_unequal))
print("Combined Fair Rewards   (penalizes both = 4.0):  ", reward_comb.calculate(obs_unequal))

# Scenario: Both buildings have the exact same discomfort (comfort loss = 2.0 each)
obs_equal = [
    dict(obs_unequal[0]),
    dict(obs_unequal[0], net_electricity_consumption=1.0),
]
print("\n=== Equal Comfort Boundary Scenario ===")
print("Disparity penalty becomes 0 when comfort is equal:", reward_disp.calculate(obs_equal))
print("Worst-case penalty still applies when comfort is equal:", reward_wc.calculate(obs_equal))

=== Unequal Comfort Scenario ===
Worst-Case Fair Rewards (penalizes max loss 2.0): [-2.0, -1.0]
Disparity-Aware Rewards (penalizes gap 2.0):      [-2.0, -1.0]
Combined Fair Rewards   (penalizes both = 4.0):   [-2.5, -1.5]

=== Equal Comfort Boundary Scenario ===
Disparity penalty becomes 0 when comfort is equal: [-1.5, -1.0]
Worst-case penalty still applies when comfort is equal: [-2.0, -1.5]


## 3. Environment Instantiation Across Candidate Controllers

Verify that environments for all 5 controllers can be created on the held-out `TEST_SPLIT` (June 26–30, 120 hours).

In [4]:
controllers = {
    "Rule-Based Control (RBC)": make_sac_env(SCHEMA, TEST_SPLIT, random_seed=42),
    "Independent SAC Baseline": make_sac_env(SCHEMA, TEST_SPLIT, random_seed=42, reward="electricity_discomfort"),
    "Standard MARLISA": make_marlisa_env(SCHEMA, TEST_SPLIT, reward="marl"),
    "Carbon-Aware MARLISA": make_marlisa_env(SCHEMA, TEST_SPLIT, reward="carbon"),
    "Worst-Case Fair MARLISA": make_marlisa_env(SCHEMA, TEST_SPLIT, reward="worst_case_fair"),
    "Disparity-Aware MARLISA": make_marlisa_env(SCHEMA, TEST_SPLIT, reward="disparity_fair"),
}

rows = []
for name, env in controllers.items():
    rows.append({
        "Controller": name,
        "Split": TEST_SPLIT.name,
        "Time Steps": env.time_steps,
        "Buildings": len(env.buildings),
        "Reward Class": type(env.reward_function).__name__,
    })
pd.DataFrame(rows)

,Controller,Split,Time Steps,Buildings,Reward Class
0,Rule-Based Control (RBC),test,120,3,ElectricityDiscomfortSACReward
1,Independent SAC Baseline,test,120,3,ElectricityDiscomfortSACReward
2,Standard MARLISA,test,120,3,MARL
3,Carbon-Aware MARLISA,test,120,3,CarbonAwareReward
4,Worst-Case Fair MARLISA,test,120,3,FairCarbonAwareReward
5,Disparity-Aware MARLISA,test,120,3,FairCarbonAwareReward


## 4. End-to-End Held-Out Evaluation Consistency Check

Execute a fast smoke training on `train` split and deterministic evaluation on `test` split to verify that all output tables, KPIs, and fairness metrics are generated consistently.

In [5]:
smoke_dir = PROJECT_ROOT / "outputs" / "pipeline_checks_smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)

# Run a short 3-episode smoke execution with Worst-Case Fair reward
result = run_marlisa(
    schema=SCHEMA,
    output_directory=smoke_dir / "marlisa_worst_case",
    episodes=3,
    random_seed=42,
    reward="worst_case_fair",
    split="train",
    batch_size=32,
    simulation_end_time_step=47,
    run_test=False,
    save_results=True,
)

print("--- Metadata Check ---")
print("Split:", result["metadata"]["split"])
print("Reward function:", result["metadata"]["reward_function"])
print("Reward weights:", result["metadata"]["reward_weights"])
print("\n--- Fairness Summary ---")
display(result["fairness"])
print("\n--- KPI Table Head ---")
display(result["kpi_table"].head())

--- Metadata Check ---
Split: train
Reward function: FairCarbonAwareReward
Reward weights: {'carbon': 1.0, 'comfort': 0.25, 'fairness': 0.25, 'worst_case': None, 'disparity': None, 'fairness_mode': 'combined', 'comfort_band_celsius': 1.0}

--- Fairness Summary ---


,average_discomfort,worst_discomfort,best_discomfort,discomfort_gap,worst_building
0,0.301552,0.432432,0.222222,0.21021,Building_2



--- KPI Table Head ---


name,Building_1,Building_2,Building_3,District
cost_function,,,,
all_time_peak_average,NaN,NaN,NaN,1.019970e+00
annual_normalized_unserved_energy_total,0.000000,4.339454e-10,0.000000,1.446485e-10
carbon_emissions_total,0.839227,7.787888e-01,0.826191,8.147357e-01
cost_total,0.828603,7.695849e-01,0.812014,8.034004e-01
daily_one_minus_load_factor_average,NaN,NaN,NaN,1.025418e+00


## Conclusion & Next Steps

The checks confirm that:
- The held-out dataset splits (`train`, `validation`, `test`) are uniformly supported across SAC and MARLISA.
- Fairness objectives are decoupled into `worst_case` (Min-Max) and `disparity` (Spread) modes.
- Output metrics, KPI tables, and checkpoints conform to the standardized reporting format ready for Week 11 sweep experiments.